# Support case assistant (Week 1 community exercise)

**Author:** dinyangetoh

## What this exercise is

This is a **Week 1 foundations** community contribution: a **manual** support agent. There is **no** OpenAI Agents SDK. You implement the classic loop yourself: `chat.completions` with tools → execute tool calls → append assistant + tool messages → repeat until the model answers in plain text or a **max iteration** cap is hit (same spirit as [4_lab4.ipynb](../../4_lab4.ipynb) and [5_extra.ipynb](../../5_extra.ipynb)).

**Learning goals:** JSON tool definitions with strong **`description`** fields; **read-only** tools (lookup order and policy) vs a **side-effect** tool (`log_escalation`); **token budget** logging after each API round (`usage`).

## What this notebook does

- **Mock CRM:** `ORDERS` spans **ORD-1001–ORD-1012** so you can stress-test refunds, cancellations, in-transit orders, **already refunded** (idempotency), **expired refund windows**, short windows, and optional fields like `item_condition_note`.
- **Mock policy:** `POLICY_DATA` holds three topics — **refund**, **shipping**, and **escalation** — with explicit rules (e.g. when to cancel vs escalate, duplicate refund handling).
- **Tools:** `get_order_summary`, `get_policy_excerpt` (topics constrained via `enum`), and `log_escalation`. Handlers live in Python; dispatch uses a **name → function** map instead of a long `if` chain.
- **Run:** `run_support_case(user_message)` prints **per-turn and cumulative** tokens and returns the assistant’s final reply; `case_notes` lists escalations written during the session.

Set `OPENAI_API_KEY` in `.env`, run cells top to bottom, then edit the example ticket or call `run_support_case(...)` with your own message.



In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv(override=True)
client = OpenAI()
MODEL = "gpt-4o-mini"


## Mocked Records for:

`ORDERS` & `POLICY`


In [ ]:
ORDERS = {
    "ORD-1001": {
        "customer_email": "buyer@example.com",
        "product": "Wireless Earbuds",
        "purchased_at": "2025-01-10",
        "delivered_at": "2025-01-14",
        "status": "delivered",
        "refund_window_days": 30,
        "amount_cents": 7999,
    },
    "ORD-1002": {
        "customer_email": "buyer@example.com",
        "product": "USB-C Hub",
        "purchased_at": "2025-02-01",
        "delivered_at": None,
        "status": "shipped",
        "refund_window_days": 14,
        "amount_cents": 4500,
    },
    # --- 10 new orders ---
    "ORD-1003": {
        "customer_email": "alice@example.com",
        "product": "Mechanical Keyboard",
        "purchased_at": "2025-01-10",
        "delivered_at": "2025-01-14",
        "status": "delivered",
        "refund_window_days": 30,
        "amount_cents": 12999,
    },
    "ORD-1004": {
        "customer_email": "bob@example.com",
        "product": "Laptop Stand",
        "purchased_at": "2025-01-05",
        "delivered_at": "2025-01-09",
        "status": "delivered",          # window expired (>30 days ago)
        "refund_window_days": 30,
        "amount_cents": 3499,
    },
    "ORD-1005": {
        "customer_email": "carol@example.com",
        "product": "Smartwatch",
        "purchased_at": "2025-02-20",
        "delivered_at": None,
        "status": "pending",            # not yet shipped
        "refund_window_days": 30,
        "amount_cents": 24999,
    },
    "ORD-1006": {
        "customer_email": "dave@example.com",
        "product": "Noise-Cancelling Headphones",
        "purchased_at": "2025-01-28",
        "delivered_at": "2025-02-03",
        "status": "refunded",           # already refunded — idempotency test
        "refund_window_days": 30,
        "amount_cents": 19999,
        "refunded_at": "2025-02-05",
    },
    "ORD-1007": {
        "customer_email": "eve@example.com",
        "product": "Portable SSD 1TB",
        "purchased_at": "2025-02-15",
        "delivered_at": None,
        "status": "cancelled",          # already cancelled
        "refund_window_days": 14,
        "amount_cents": 8999,
    },
    "ORD-1008": {
        "customer_email": "frank@example.com",
        "product": "Monitor 27\"",
        "purchased_at": "2025-02-12",
        "delivered_at": "2025-02-18",
        "status": "delivered",
        "refund_window_days": 14,       # short window — good edge case
        "amount_cents": 34999,
    },
    "ORD-1009": {
        "customer_email": "grace@example.com",
        "product": "Webcam 4K",
        "purchased_at": "2025-02-22",
        "delivered_at": "2025-02-26",
        "status": "delivered",
        "refund_window_days": 30,
        "amount_cents": 9999,
    },
    "ORD-1010": {
        "customer_email": "heidi@example.com",
        "product": "Ergonomic Mouse",
        "purchased_at": "2025-01-15",
        "delivered_at": "2025-01-20",
        "status": "delivered",          # borderline: ~30 days from delivery
        "refund_window_days": 30,
        "amount_cents": 5999,
    },
    "ORD-1011": {
        "customer_email": "ivan@example.com",
        "product": "USB Microphone",
        "purchased_at": "2025-02-25",
        "delivered_at": None,
        "status": "shipped",
        "refund_window_days": 30,
        "amount_cents": 7499,
    },
    "ORD-1012": {
        "customer_email": "judy@example.com",
        "product": "Drawing Tablet",
        "purchased_at": "2025-02-18",
        "delivered_at": "2025-02-23",
        "status": "delivered",
        "refund_window_days": 30,
        "amount_cents": 15999,
        "item_condition_note": "customer reports screen scratches on arrival",
    },
}

print("ORDERS loaded")

## POLICY

In [ ]:
POLICY_DATA = {
    "refund": (
        "Refunds: a refund is eligible if the request is made within the order's "
        "refund_window_days after the delivered_at date, and the item is defective or "
        "not as described. "
        "If the order has not yet been delivered (status = 'shipped' or 'pending'), "
        "the customer may cancel for a full refund. "
        "If the order is already in status 'refunded', inform the customer the refund "
        "was already processed and provide the refunded_at date if available; do not "
        "initiate a duplicate refund. "
        "If the order is already 'cancelled', inform the customer and take no further action. "
        "If the refund window has expired or the reason does not qualify, escalate to "
        "Tier-2 for discretionary review."
    ),
    "shipping": (
        "Shipping: standard delivery takes 5–7 business days after purchase. "
        "Orders in 'pending' status have not yet been picked up by the carrier. "
        "Orders in 'shipped' status are in transit; the customer should wait for delivery "
        "before requesting a replacement or refund unless significant delay is suspected."
    ),
    "escalation": (
        "Escalation: log a Tier-2 ticket whenever: (1) refund window expired, "
        "(2) reason given does not clearly match policy, "
        "(3) order amount exceeds 20000 cents and any doubt exists, "
        "or (4) customer disputes a decision. "
        "Include order ID, customer email, stated reason, and agent decision summary."
    ),
}

print("POLICY data loaded")

## Tool Handlers


In [ ]:
case_notes: list[str] = []

def get_order_summary(order_id: str) -> dict:
    """Lookup used by tool — read-only."""
    row = ORDERS.get(order_id)
    if not row:
        return {"error": f"unknown_order_id: {order_id}"}
    return {"order_id": order_id, **row}


def get_policy_excerpt(topic: str) -> dict:
    """Return canned policy text — read-only."""
    key = topic.lower().strip()
    text = POLICY_DATA.get(key)
    if not text:
        return {"topic": topic, "excerpt": "No snippet for this topic. Try topics: refund, shipping, escalation."}
    return {"topic": topic, "excerpt": text}


def log_escalation(reason: str, order_id: str) -> dict:
    """Append escalation — side effect (mutable state)."""
    line = f"[{order_id}] {reason}"
    case_notes.append(line)
    return {"logged": True, "case_note": line, "total_notes": len(case_notes)}

print("Tool handlers loaded")

## Tool call schemas


In [ ]:
get_order_summary_json = {
    "name": "get_order_summary",
    "description": (
        "Read-only. Returns order facts: status, purchased_at, delivered_at, "
        "refund_window_days, amount_cents, and any condition notes. "
        "Always call this first — before get_policy_excerpt and before log_escalation. "
        "Do not reason about refund eligibility without calling this first."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "order_id": {
                "type": "string",
                "description": "Order ID exactly as given by the customer, e.g. ORD-1001.",
            },
        },
        "required": ["order_id"],
        "additionalProperties": False,
    },
}

get_policy_excerpt_json = {
    "name": "get_policy_excerpt",
    "description": (
        "Read-only. Returns the policy text for a given topic. "
        "Available topics: 'refund' (eligibility, windows, idempotency), "
        "'shipping' (delivery timelines, status semantics), "
        "'escalation' (when and how to escalate to Tier-2). "
        "Call this after get_order_summary and before deciding whether to resolve or escalate."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "enum": ["refund", "shipping", "escalation"],
                "description": "Policy topic to retrieve.",
            },
        },
        "required": ["topic"],
        "additionalProperties": False,
    },
}

log_escalation_json = {
    "name": "log_escalation",
    "description": (
        "Side effect — writes a Tier-2 review ticket. "
        "Only call this when policy explicitly requires Tier-2 judgment "
        "(e.g. expired window, ambiguous reason, high-value order with doubt). "
        "Precondition: get_order_summary must have been called in this turn. "
        "The reason field must reference specific order facts, not generic text."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "order_id": {
                "type": "string",
                "description": "Order ID being escalated, e.g. ORD-1008.",
            },
            "reason": {
                "type": "string",
                "description": (
                    "Specific reason Tier-2 is needed. Must cite order facts: "
                    "e.g. 'Refund window expired — delivered 2025-02-03, requested 2025-03-10, "
                    "window is 14 days. Customer claims item defective.'"
                ),
            },
        },
        "required": ["order_id", "reason"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": get_order_summary_json},
    {"type": "function", "function": get_policy_excerpt_json},
    {"type": "function", "function": log_escalation_json},
]

print("Tool call schemas loaded")

## Tool call dispatcher


In [ ]:
# TOOL_IMPL = {
#     "get_order_summary": get_order_summary,
#     "get_policy_excerpt": get_policy_excerpt,
#     "log_escalation": log_escalation,
# }


def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments or "{}")
        tool = globals().get(name)
        print(f"Tool: {name} | Args: {args}")
        result = tool(**args) if tool else {"error": f"unknown_tool:{name}"}
        results.append(
            {"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id}
        )
    return results

print("Tool dispatcher loaded")

## Agent loop + token accounting


In [ ]:
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")
print(f"Today's date: {today}")

SYSTEM_PROMPT = f"""You are a Tier-1 support agent for an e-commerce store. 
Your job is to resolve customer tickets accurately using tools, not memory.

## Decision flow
1. Call get_order_summary with the customer's order ID to read facts.
2. Call get_policy_excerpt with the relevant topic (refund, shipping, or escalation) to read rules.
3. Apply policy to facts. Then either:
   a. Resolve: reply concisely — state your decision and the specific policy reason.
   b. Escalate: call log_escalation with a precise reason, then inform the customer their case is escalated.

## Hard rules
- Never invent or assume order data. If the customer gives no order ID, ask for it before doing anything else.
- Never call log_escalation without first calling get_order_summary.
- If the order is already refunded or cancelled, inform the customer of that fact — do not take further action.
- Keep replies short. Customers do not need to see policy text verbatim.
- Before deciding refund eligibility, explicitly calculate:
  days_since_delivery = today - delivered_at
  Then compare to refund_window_days.
  Never declare a refund eligible without stating this calculation.
- Today's date is {today}.
"""

print(SYSTEM_PROMPT)

def get_token_usage(iter_no,response):
    token_usage = response.usage
    if token_usage:
        total_prompt += token_usage.prompt_tokens or 0
        total_completion += token_usage.completion_tokens or 0
        total_tokens += token_usage.total_tokens or 0
        print(
            f"Iteration {iter_no + 1} tokens: prompt={token_usage.prompt_tokens} completion={token_usage.completion_tokens} "
            f"total={token_usage.total_tokens} | cumulative_total={total_tokens}"
        )
    return total_prompt, total_completion, total_tokens

def run_support_case(user_message: str, max_iterations: int = 5):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    total_prompt = total_completion = total_tokens = 0
    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )
        token_usage = response.usage

        if token_usage:
            total_prompt += token_usage.prompt_tokens or 0
            total_completion += token_usage.completion_tokens or 0
            total_tokens += token_usage.total_tokens or 0
            print(
                f"Iteration {iteration + 1} tokens: prompt={token_usage.prompt_tokens} completion={token_usage.completion_tokens} "
                f"total={token_usage.total_tokens} | cumulative_total={total_tokens}"
            )
        choice = response.choices[0]
        if choice.finish_reason == "tool_calls" and choice.message.tool_calls:
            messages.append(choice.message)
            messages.extend(handle_tool_calls(choice.message.tool_calls))
            continue
        return choice.message.content or "", total_tokens
    return "Stopped: max_iterations reached (no final assistant text).", total_tokens


## Example ticket

Edit `user_message` to experiment.


In [ ]:
case_notes.clear()

user_message = (
    "Hi, my order ORD-1001 arrived but one earbud is dead. "
    "I want a refund — it's been 20 days since delivery."
)

answer, cumulative = run_support_case(user_message)
print("\n=== Final reply ===\n", answer)
print("\n=== Escalation log ===\n", case_notes or "(none)")


In [ ]:
user_message = (
    "Hi, my order ORD-1002, I would like to cancel my order. "
)

answer, cumulative = run_support_case(user_message)
print("\n=== Final reply ===\n", answer)
print("\n=== Escalation log ===\n", case_notes or "(none)")